# 02 - Limpieza de datos

## Objetivo de este cuaderno

Aquí aplicamos, una por una, las decisiones tomadas en el EDA. Partimos siempre del CSV original, que nunca se modifica, y al final guardamos el resultado limpio.

- **Lee:** data/raw/coches_segunda_mano.csv
- **Guarda:** data/processed/coches_limpio.csv

### Importar librerías

Cargamos pandas, numpy, matplotlib y seaborn, que usaremos en todo el análisis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Cargar el dataset

Leemos el CSV de anuncios y mostramos las primeras filas para comprobar que la carga es correcta.

In [2]:
df = pd.read_csv('../data/raw/coches_segunda_mano.csv')
df.head()

,url,company,make,model,version,price,price_financed,fuel,year,kms,...,doors,shift,color,photos,is_professional,dealer,province,country,publish_date,insert_date
0,e158ae0ca53119ca199c28c36b5c2fcd,9881bcdd5a0ad4733037b3fb25e69c3a,SEAT,Toledo,SEAT Toledo 4p.,950,NaN,Diésel,2000.0,227000,...,4,Manual,Verde,5,False,0f4bb8455d27349b8273109b66a847f3,Navarra,Spain,2020-12-18 10:47:13,2021-01-15 00:00:00
1,ff267ebb7e700246f47f84f3db660b4b,9881bcdd5a0ad4733037b3fb25e69c3a,CITROEN,C1,CITROEN C1 PureTech 60KW 82CV Feel 5p.,6200,NaN,Gasolina,2017.0,50071,...,5,Manual,Blanco,6,True,Autos Raymara,Tenerife,Spain,2021-01-02 11:25:40,2021-01-15 00:00:00
2,de4b02db28ea7786c622b969be10c7c7,9881bcdd5a0ad4733037b3fb25e69c3a,FORD,Transit Connect,FORD Transit Connect Van 1.5 TDCi 100cv Ambien...,7851,7024.0,Diésel,2016.0,103000,...,4,Manual,Blanco,10,True,Auto 96,Barcelona,Spain,2020-12-16 10:51:45,2021-01-15 00:00:00
3,0449972a4d07594acf92e9a7dd28b39c,9881bcdd5a0ad4733037b3fb25e69c3a,VOLKSWAGEN,Caravelle,VOLKSWAGEN Caravelle Largo 2.0 TDI 140 Comfort...,19426,NaN,Diésel,2014.0,120000,...,4,Manual,Blanco,9,True,Inniauto,Navarra,Spain,2020-11-25 11:09:14,2021-01-15 00:00:00
4,12c4fa49bd4fdf23f19ecf396d3f02ef,9881bcdd5a0ad4733037b3fb25e69c3a,FORD,Transit,FORD Transit 350 96kW L4 Ambiente Propulsion T...,22850,22800.0,Diésel,2017.0,107000,...,2,Manual,Blanco,4,True,"Autofleet España,s.l",Sevilla,Spain,2021-01-12 20:00:34,2021-01-15 00:00:00


### Comprobación: ¿tienen gemelos los coches sin potencia?

A 8.528 anuncios les falta la potencia. Antes de decidir cómo rellenarla, medimos si se puede copiar de **coches iguales** que sí la tienen:

1. Primero, coches con la misma marca, modelo y versión (la versión suele fijar el motor).
2. Si no hay, coches con la misma marca y modelo (menos preciso, porque un modelo puede tener varios motores).

Aquí **solo medimos**. El relleno, si se hace, se aplicará en 02_limpieza. El df original no se modifica: trabajamos con copias.

Separamos los anuncios sin potencia y con potencia.

In [3]:
sin_power = df[df['power'].isnull()].copy()
con_power = df[df['power'].notnull()].copy()
print('Sin potencia:', len(sin_power))
print('Con potencia:', len(con_power))

Sin potencia: 8528
Con potencia: 41472


Creamos una clave de texto con marca, modelo y versión, y otra solo con marca y modelo, para poder comparar.

In [4]:
sin_power['clave_mmv'] = sin_power['make'] + ' | ' + sin_power['model'] + ' | ' + sin_power['version']
con_power['clave_mmv'] = con_power['make'] + ' | ' + con_power['model'] + ' | ' + con_power['version']
sin_power['clave_mm'] = sin_power['make'] + ' | ' + sin_power['model']
con_power['clave_mm'] = con_power['make'] + ' | ' + con_power['model']

Contamos cuántos coches sin potencia tienen un gemelo exacto (marca + modelo + versión) con potencia.

In [5]:
gemelo_mmv = sin_power['clave_mmv'].isin(con_power['clave_mmv'])
print('Con gemelo marca + modelo + versión:', gemelo_mmv.sum())

Con gemelo marca + modelo + versión: 88


De los que no tienen gemelo exacto, contamos cuántos tienen al menos un coche de la misma marca y modelo con potencia.

In [6]:
restantes = sin_power[~gemelo_mmv]
gemelo_mm = restantes['clave_mm'].isin(con_power['clave_mm'])
print('Restantes sin gemelo exacto:', len(restantes))
print('De ellos, con gemelo marca + modelo:', gemelo_mm.sum())
print('Sin ningún gemelo:', len(restantes) - gemelo_mm.sum())

Restantes sin gemelo exacto: 8440
De ellos, con gemelo marca + modelo: 8332
Sin ningún gemelo: 108


Miramos algunos ejemplos de versión de los coches sin potencia, para entender por qué no encuentran gemelo exacto.

In [7]:
sin_power['version'].value_counts().head(10)

version
BMW              399
VOLKSWAGEN       392
RENAULT          279
MERCEDES-BENZ    266
AUDI             264
PEUGEOT          258
FORD             249
SEAT             248
OPEL             243
CITROEN          191
Name: count, dtype: int64

Por último, contamos cuántos nulos de potencia quedarían si aplicamos los cortes decididos (precio entre 1.000 € y 100.000 €, hasta 1.000.000 km, potencia hasta 700 CV o vacía, y coches de 2006 en adelante). Usamos una copia: df no cambia.

In [8]:
df_cortes = df[(df['price'] >= 1000) & (df['price'] <= 100000) & (df['kms'] <= 1000000) & ((df['power'] <= 700) | (df['power'].isnull())) & (df['year'] >= 2006)]
print('Anuncios tras los cortes:', len(df_cortes))
print('Nulos de potencia tras los cortes:', df_cortes['power'].isnull().sum())

Anuncios tras los cortes: 40801
Nulos de potencia tras los cortes: 3842


### Observaciones (gemelos de los coches sin potencia)

- De los **8.528** anuncios sin potencia, solo **88** (1 %) tienen un gemelo exacto con la misma marca, modelo y versión.
- El motivo se ve en los ejemplos: en muchos de estos anuncios la versión no describe el motor. A menudo es solo la marca ("BMW", "VOLKSWAGEN"...) o marca, modelo y puertas ("RENAULT Laguna 5p."). Por eso casi nunca coinciden con una versión completa.
- De los 8.440 restantes, **8.332** sí tienen coches de la misma marca y modelo con potencia. Solo **108** no tienen ningún gemelo.
- Ojo: un mismo modelo puede llevar motores muy distintos (un Serie 3 puede tener de 116 a más de 400 CV), así que rellenar con la marca y el modelo es una aproximación, no un dato exacto.
- Tras aplicar los cortes decididos (precio de 1.000 € a 100.000 €, hasta 1.000.000 km, potencia hasta 700 CV y coches de 2006 en adelante) quedan **40.801** anuncios y **3.842** siguen sin potencia (el 9,4 %). Los cortes, sobre todo el de antigüedad, eliminan más de la mitad de los nulos, porque se concentraban en coches viejos.
- **Para 02_limpieza:** valorar rellenar la potencia con la mediana de la misma marca y modelo, y decidir qué hacer con los pocos que no tienen gemelo.

### Eliminar el precio financiado

*Decisión tomada en el EDA (ver 01_eda.ipynb).*

La quitamos por dos motivos: falta en más de la mitad de los anuncios y, donde existe, va casi a la par que price (correlación ≈ 0,997), así que no aporta información nueva. power se queda como está y se trata más adelante. El resto de huecos, que son pocos, se revisa de uno en uno.

In [9]:
df = df.drop(columns='price_financed')

### Marca de los D2S

*Decisión tomada en el EDA (ver 01_eda.ipynb).*

Las dos filas sin marca tienen modelo D2S y no hay otro anuncio del que copiarla. Buscado fuera del dataset, corresponde a Invicta Electric (fuente: web oficial del fabricante, https://invictaelectric.es/tienda/d2s-17kw/, y ABC Motor, https://www.abc.es/motor/novedades/abci-invicta-coche-acceso-movilidad-electrica-202106170100_noticia.html). Escribimos la marca en mayúsculas, como el resto.

In [10]:
df.loc[df['make'].isnull() & (df['model'] == 'D2S'), 'make'] = 'INVICTA ELECTRIC'
df.loc[df['model'] == 'D2S', ['make', 'model', 'version']]

,make,model,version
5659,INVICTA ELECTRIC,D2S,D2S 27kW 3p.
47381,INVICTA ELECTRIC,D2S,D2S 27kW 3p.


## Pasos pendientes

- Quitar url y company (y decidir sobre country y dealer).
- Revisar los 5 nulos de model y el resto de huecos pequeños.
- Convertir year a entero (int16) tras tratar sus 2 nulos. El año no tiene decimales; int16 basta y reduce memoria (int64 ocupa lo mismo que float64).
- Pasar las fechas a tipo fecha si se usan.
- Unificar colores.
- Guardar el resultado en data/processed/coches_limpio.csv.